# Notebook 08 — Position Encodings and RoPE

    ## Learning objectives

    - Explain why attention alone is permutation equivariant
- Compare learned, sinusoidal, relative, ALiBi, and rotary position methods
- Implement RoPE and reason about context extension

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from the Colab Secrets UI without displaying it. Create a
# secret named exactly HF_TOKEN and enable notebook access with its toggle.
token = os.getenv("HF_TOKEN")
token_error = None
if IN_COLAB and not token:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        token_error = type(exc).__name__
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub environment variable.
if token:
    os.environ["HF_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("HF_TOKEN is unavailable. In Colab, open the key icon (Secrets), add HF_TOKEN, ")
    print("enable its Notebook access toggle, and rerun this cell. Public models still work.")
    if token_error:
        print("Colab secret lookup status:", token_error)


## 8.1 Position must enter somewhere

Without position information, permuting input tokens permutes outputs but does not
change the attention relation itself. Learned absolute embeddings add a position vector.
Sinusoids use fixed frequencies. Relative schemes bias attention by token distance.
ALiBi adds head-specific linear distance penalties. RoPE rotates query/key pairs so
their dot product depends on relative displacement.


## 8.2 Rotary position embedding

For each two-dimensional feature pair and angle \(m\theta_i\), apply

\[
R(m\theta_i)\begin{bmatrix}x_{2i}\\x_{2i+1}\end{bmatrix}.
\]

Rotating Q and K at positions \(m,n\) makes their inner product contain
\(R((n-m)\theta)\), encoding relative position. Frequencies are geometrically spaced.
Values are not rotated.


In [ ]:
import torch

def rope(x, positions, base=10_000):
    # x: [..., T, D], D even
    D = x.shape[-1]
    inv_freq = base ** (-torch.arange(0, D, 2, device=x.device) / D)
    angles = positions[:, None] * inv_freq[None, :]
    cos, sin = angles.cos(), angles.sin()
    even, odd = x[..., 0::2], x[..., 1::2]
    return torch.stack((even * cos - odd * sin,
                        even * sin + odd * cos), dim=-1).flatten(-2)

q = torch.randn(1, 6, 8)
pos = torch.arange(6)
rotated = rope(q, pos)
print(rotated.shape)
print("norm preserved:", torch.allclose(q.norm(dim=-1), rotated.norm(dim=-1), atol=1e-5))


## 8.3 Context extension is not free

RoPE extrapolation can degrade when inference positions exceed training positions.
Scaling methods alter positions or frequencies, but a larger configured window does
not prove useful long-context behavior. Evaluate retrieval at different depths,
instruction following, and perplexity across positions. Long context also enlarges KV
memory and prefill compute even when positional quality holds.


## 8.4 Comparing positional strategies

Learned absolute embeddings are simple and flexible inside their trained range, but have a
fixed table and poor extrapolation. Sinusoidal encodings require no learned table and expose
multiple wavelengths, yet add position to token state rather than attention relations.
Relative position biases directly modify attention scores by displacement buckets. ALiBi
uses a monotonic, head-specific distance penalty and extrapolates operationally without a
table. RoPE rotates Q/K features and has become common in decoder LLMs. Some architectures
mix sliding/local attention with periodic global layers, changing what “position handling”
means at long range.

There is no context extension switch independent of training. Interpolation compresses new
positions into the trained range; NTK-aware and frequency-selective variants adjust RoPE
frequencies; YaRN-like approaches combine scaling and attention adjustments. Each changes
the distribution the model sees. Long-context continued training may be required, and
evaluation must test more than a single retrieval needle.


In [ ]:
# Visualize RoPE wavelengths and rotations across positions.
import matplotlib.pyplot as plt
D = 16
inv_freq = 10_000 ** (-torch.arange(0, D, 2) / D)
positions = torch.arange(0, 256)
angles = positions[:, None] * inv_freq[None, :]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(inv_freq.numpy(), marker="o")
axes[0].set(title="RoPE inverse frequencies", xlabel="feature pair", ylabel="radians/position")
for i in [0, 2, 4, 7]: axes[1].plot(positions, angles[:, i].cos(), label=f"pair {i}")
axes[1].set(title="Cosine phase by position", xlabel="position"); axes[1].legend()
plt.tight_layout()


## 8.5 Relative-position identity and implementation details

Rotation matrices obey \(R(a)^T R(b)=R(b-a)\). Therefore
\((R(m)q)^T(R(n)k)=q^T R(n-m)k\): the attention dot product exposes relative displacement
while each vector also carries absolute phase. Implementations often use a “rotate half”
arrangement rather than adjacent feature pairs; both are valid only when frequencies and
layout match model training. RoPE is applied after Q/K projection, commonly to only a rotary
sub-dimension, and must use absolute cache positions during incremental decoding.

Cache correctness is a frequent failure: if a new token is rotated as position zero rather
than its actual sequence index, cached generation diverges. Padding requires explicit
position IDs in some batching layouts. Packed sequences may reset positions per example or
continue monotonically, depending on training design. Changing base, scaling, rotary
percentage, or layout makes existing weights incompatible even though tensor shapes match.


In [ ]:
# Numerically verify that jointly shifting q/k positions preserves their RoPE dot product.
torch.manual_seed(3)
q, k = torch.randn(1, 1, 8), torch.randn(1, 1, 8)
def score(q_pos, k_pos):
    qr = rope(q, torch.tensor([q_pos]))
    kr = rope(k, torch.tensor([k_pos]))
    return float((qr * kr).sum())
for shift in [0, 5, 100]:
    print(shift, score(7 + shift, 13 + shift))
print("different displacement:", score(7, 14))


## 8.6 Long-context evaluation reference

Test several capabilities and positions: exact retrieval, multi-hop synthesis across distant
passages, aggregation over many records, instruction persistence, conflicting evidence,
recent versus early evidence, and generation after a long prefill. Include distractors that
share vocabulary with the question. Measure accuracy by depth and total length, TTFT,
memory, and tokens/second. Inspect whether failures come from truncation, retrieval,
attention, generation, or the evaluation itself.

“Needle in a haystack” is a diagnostic, not a complete benchmark: exact distinctive strings
can be matched without robust comprehension. Long-context RAG may still outperform placing
everything in context because retrieval filters distraction and reduces compute. Conversely,
retrieval can omit decisive evidence. Choose architecture using representative tasks and
end-to-end cost, not maximum context length on a model card.


## 8.7 Positional-method reference

| Method | Injected where | Learned? | Extrapolation considerations |
|---|---|---:|---|
| Learned absolute | Added to hidden state | Yes | Table/range-bound |
| Sinusoidal | Added to hidden state | No | Defined beyond training; behavior not guaranteed |
| Relative bias | Attention logits | Often | Bucketing/saturation policy matters |
| ALiBi | Attention logits | Slopes often fixed | Distance penalty supports longer indices |
| RoPE | Rotates Q/K | No frequencies, sometimes scaling config | Phase/frequency distribution shifts |

Do not conflate an implementation accepting longer position IDs with a model reasoning effectively at
that length. Verify tokenizer/model maximums, RoPE cache growth, position IDs under padding/packing,
KV memory, and attention backend support. Changing RoPE base or scaling after training is an inference
intervention and deserves held-out quality tests across lengths and positions.

Useful diagnostics include joint-shift invariance, norm preservation, cached-versus-full logits,
retrieval by depth, multi-hop evidence separation, long-output stability, and perplexity versus token
position. Always compare against a shorter-context/RAG baseline with equal answer evidence.


## 8.8 RoPE preserves relative phase structure

RoPE groups hidden dimensions into pairs and rotates each pair by a position-dependent angle. The dot product between rotated queries and keys depends on their relative position because the absolute rotations combine into a rotation by the position difference. Test this property numerically by shifting both token positions by the same offset. The result should remain close for an unmodified frequency schedule. This does not imply unlimited context: learned attention behavior, numerical phase resolution, training lengths, and frequency scaling still matter. Always rotate queries and keys consistently while leaving values unchanged.


In [ ]:
def rotate_pair(x,position,theta=.3):
 angle=position*theta; c,s=torch.cos(torch.tensor(angle)),torch.sin(torch.tensor(angle)); R=torch.tensor([[c,-s],[s,c]])
 return x@R.T
q=torch.tensor([.8,-.2]); k=torch.tensor([.1,.9])
a=rotate_pair(q,3)@rotate_pair(k,8); b=rotate_pair(q,13)@rotate_pair(k,18)
print(a.item(),b.item()); torch.testing.assert_close(a,b)


## 8.9 Context extension is an evaluation problem

Interpolation, frequency rescaling, NTK-aware variants, and methods such as YaRN modify how positions map to rotation frequencies. Their names do not guarantee quality beyond the training window. Evaluate retrieval at controlled depths, repeated-token disambiguation, long-document likelihood, generation stability, and the actual application distribution. Compare with truncation, chunked retrieval, recurrence, or a model trained for the target context. Measure memory and latency because attention and KV-cache costs still grow with sequence length even when positional encoding accepts larger indices. Pin the exact scaling configuration with the checkpoint.


In [ ]:
strategies=[{"name":"truncate","context":4096,"quality":.82,"kv_gib":1.0},{"name":"scaled_rope","context":16384,"quality":.76,"kv_gib":4.0},{"name":"rag","context":4096,"quality":.88,"kv_gib":1.0}]
for row in strategies: print(row["name"],"quality/GiB",row["quality"]/row["kv_gib"])


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [RoFormer / RoPE](https://arxiv.org/abs/2104.09864)
- [YaRN](https://arxiv.org/abs/2309.00071)


## Exercises

    1. Show algebraically that rotating both vectors preserves same-position dot products.
2. Visualize low- and high-frequency RoPE dimensions across 2,048 positions.
3. Design a needle-in-a-haystack test that cannot be passed using lexical shortcuts.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
